In [ ]:
# Install required dependencies
!pip install -q kaggle-benchmarks numpy

# Metacognitive Monitoring During LearningCross-domain benchmark testing metacognition while learning.**Cognitive Science**: Dunlosky & Nelson (1992), Zimmerman (2000)

In [ ]:
"""Novel Rule System Generator for Learning Benchmarks.Generates procedural rule systems that cannot be in training data.Each system defines a mapping from inputs to outputs via a chainof deterministic rules. Difficulty is controlled by:- Number of rules- Number of input features- Rule interaction complexity (independent vs. chained)Systems are seeded for reproducibility across runs."""import randomimport hashlibfrom dataclasses import dataclass, field@dataclassclass RuleSystem:    """A generated rule system with examples."""    name: str    description: str    rules: list[str]    examples: list[dict]  # {"input": str, "output": str}    test_items: list[dict]  # {"input": str, "output": str}    difficulty: int  # 1-3    n_rules: int    domain: str  # "symbol", "language", "number"def _make_rng(seed: str) -> random.Random:    h = int(hashlib.sha256(seed.encode()).hexdigest(), 16)    return random.Random(h)def generate_symbol_system(seed: str = "sym_default", difficulty: int = 1) -> RuleSystem:    """    Generate a symbol transformation rule system.    Input: sequence of symbols (e.g., "△ ○ □")    Rules: transformations (e.g., "△ followed by ○ becomes ★")    Output: transformed sequence    """    rng = _make_rng(seed)    shapes = ["△", "○", "□", "◇", "★", "⬡", "⬟", "▽"]    colors = ["red", "blue", "green", "yellow"]    if difficulty == 1:        # Simple 1-to-1 substitution        src = rng.sample(shapes[:4], 3)        dst = rng.sample(shapes[4:], 3) + [rng.choice(shapes[4:])]        mapping = dict(zip(src, dst[:3]))        rules = [f"Replace {s} with {d}" for s, d in mapping.items()]        rules.append("All other symbols stay the same")        def apply_rules(seq):            return [mapping.get(s, s) for s in seq]    elif difficulty == 2:        # Context-dependent: pairs matter        src = rng.sample(shapes[:5], 4)        dst = rng.sample(shapes[4:], 3) + [rng.choice(shapes)]        mapping = dict(zip(src[:3], dst[:3]))        pair_rule = (src[0], src[1], dst[3])  # "X followed by Y becomes Z"        rules = [f"Replace {s} with {d}" for s, d in mapping.items()]        rules.append(f"EXCEPTION: {pair_rule[0]} followed by {pair_rule[1]} → both become {pair_rule[2]}")        rules.append("All other symbols stay the same")        def apply_rules(seq):            result = []            i = 0            while i < len(seq):                if i + 1 < len(seq) and seq[i] == pair_rule[0] and seq[i + 1] == pair_rule[1]:                    result.extend([pair_rule[2], pair_rule[2]])                    i += 2                else:                    result.append(mapping.get(seq[i], seq[i]))                    i += 1            return result    else:  # difficulty == 3        # Multi-pass with conditional rules        src = rng.sample(shapes[:6], 5)        dst = rng.sample(shapes, 5)        mapping1 = {src[0]: dst[0], src[1]: dst[1]}        mapping2 = {dst[0]: dst[2]}  # Chain: src[0] → dst[0] → dst[2]        cond = src[2]  # If this symbol is present, apply extra rule        extra_map = {src[3]: dst[3]}        rules = [            f"Pass 1: Replace {s} with {d}" for s, d in mapping1.items()        ]        rules.append(f"Pass 2: Replace {list(mapping2.keys())[0]} with {list(mapping2.values())[0]}")        rules.append(f"IF the sequence contains {cond}: also replace {src[3]} with {dst[3]}")        rules.append("All other symbols stay the same throughout")        def apply_rules(seq):            # Pass 1            result = [mapping1.get(s, s) for s in seq]            # Pass 2            result = [mapping2.get(s, s) for s in result]            # Conditional            if cond in seq:  # Check original sequence                result = [extra_map.get(s, s) for s in result]            return result    # Generate examples    all_items = []    for _ in range(25):        length = rng.randint(3, 6)        seq = [rng.choice(shapes[:5]) for _ in range(length)]        output = apply_rules(seq)        all_items.append({"input": " ".join(seq), "output": " ".join(output)})    # Deduplicate by input    seen = set()    unique_items = []    for item in all_items:        if item["input"] not in seen:            seen.add(item["input"])            unique_items.append(item)    rng.shuffle(unique_items)    n_examples = min(15, len(unique_items) - 5)    examples = unique_items[:n_examples]    test_items = unique_items[n_examples:n_examples + 5]    return RuleSystem(        name=f"SymbolTransform-{seed}",        description="Apply symbol transformation rules to input sequences",        rules=rules,        examples=examples,        test_items=test_items,        difficulty=difficulty,        n_rules=len(rules),        domain="symbol",    )def generate_number_system(seed: str = "num_default", difficulty: int = 1) -> RuleSystem:    """    Generate a novel number system / arithmetic.    Input: expression in the invented system    Rules: how operators work    Output: numeric result    """    rng = _make_rng(seed)    op_names = ["grok", "flim", "zorp", "quex", "blix"]    ops = rng.sample(op_names, 3)    if difficulty == 1:        # Two operators: basic arithmetic with twist        a_op, b_op = ops[0], ops[1]        a_fn = lambda x, y: x + y + 1  # "grok" = add and increment        b_fn = lambda x, y: abs(x - y)  # "flim" = absolute difference        rules = [            f"'{a_op}(x, y)' means: add x and y, then add 1",            f"'{b_op}(x, y)' means: absolute difference of x and y",        ]        op_map = {a_op: a_fn, b_op: b_fn}    elif difficulty == 2:        a_op, b_op, c_op = ops[0], ops[1], ops[2]        a_fn = lambda x, y: x * 2 + y        b_fn = lambda x, y: (x + y) % 10        c_fn = lambda x, y: max(x, y) - min(x, y) + 1        rules = [            f"'{a_op}(x, y)' means: double x, then add y",            f"'{b_op}(x, y)' means: add x and y, take the last digit (mod 10)",            f"'{c_op}(x, y)' means: difference of larger and smaller, plus 1",        ]        op_map = {a_op: a_fn, b_op: b_fn, c_op: c_fn}    else:  # difficulty == 3        a_op, b_op, c_op = ops[0], ops[1], ops[2]        # Nested operations        a_fn = lambda x, y: x + y + 1        b_fn = lambda x, y: x * y        rules = [            f"'{a_op}(x, y)' means: add x and y, then add 1",            f"'{b_op}(x, y)' means: multiply x and y",            f"Operations can be nested: '{a_op}({b_op}(x, y), z)' means: first compute {b_op}(x, y), then use the result as the first argument to {a_op}",        ]        op_map = {a_op: a_fn, b_op: b_fn}    # Generate examples    all_items = []    for _ in range(20):        if difficulty <= 2:            op_name = rng.choice(list(op_map.keys()))            x = rng.randint(1, 9)            y = rng.randint(1, 9)            result = op_map[op_name](x, y)            expr = f"{op_name}({x}, {y})"        else:            # Allow nesting            if rng.random() < 0.5:                op_name = rng.choice(list(op_map.keys()))                x = rng.randint(1, 9)                y = rng.randint(1, 9)                result = op_map[op_name](x, y)                expr = f"{op_name}({x}, {y})"            else:                inner_op = rng.choice(list(op_map.keys()))                outer_op = rng.choice(list(op_map.keys()))                x, y, z = rng.randint(1, 5), rng.randint(1, 5), rng.randint(1, 5)                inner_result = op_map[inner_op](x, y)                result = op_map[outer_op](inner_result, z)                expr = f"{outer_op}({inner_op}({x}, {y}), {z})"        all_items.append({"input": expr, "output": str(result)})    # Deduplicate    seen = set()    unique_items = []    for item in all_items:        if item["input"] not in seen:            seen.add(item["input"])            unique_items.append(item)    rng.shuffle(unique_items)    n_ex = min(12, len(unique_items) - 5)    examples = unique_items[:n_ex]    test_items = unique_items[n_ex:n_ex + 5]    return RuleSystem(        name=f"NumberSystem-{seed}",        description="Evaluate expressions using novel arithmetic operators",        rules=rules,        examples=examples,        test_items=test_items,        difficulty=difficulty,        n_rules=len(rules),        domain="number",    )# Pre-generated systems for the benchmarkLEARNING_CURVE_SYSTEMS = [    generate_symbol_system("lc_sym_easy", difficulty=1),    generate_symbol_system("lc_sym_med", difficulty=2),    generate_symbol_system("lc_sym_hard", difficulty=3),    generate_number_system("lc_num_easy", difficulty=1),    generate_number_system("lc_num_med", difficulty=2),    generate_number_system("lc_num_hard", difficulty=3),]# Systems for transfer testingTRANSFER_BASE_SYSTEM = generate_symbol_system("transfer_base", difficulty=2)TRANSFER_NEAR_SYSTEM = generate_symbol_system("transfer_near", difficulty=2)TRANSFER_FAR_SYSTEM = generate_number_system("transfer_far", difficulty=2)# Systems for interference testingINTERFERENCE_A = generate_symbol_system("interf_a", difficulty=2)INTERFERENCE_B = generate_symbol_system("interf_b_similar", difficulty=2)

In [ ]:
"""Cross-Domain Benchmark: Metacognitive Monitoring During LearningThis is a unique benchmark that tests metacognition and learning simultaneously.Rather than measuring them in isolation, this tests whether a model canaccurately monitor its own learning process in real-time.Protocol:1. Present novel rule system incrementally (one rule at a time)2. After each new rule, ask TWO things:   a. Apply what you've learned so far (learning test)   b. Rate how well you think you've learned the system so far (metacognitive probe)3. Measure: Does self-assessment track actual learning curve?This is grounded in the "calibration of learning" literature:- Dunlosky & Nelson (1992): Metacognitive monitoring during learning- Koriat (1997): Cue-utilization in JOLs during acquisition- Zimmerman (2000): Self-regulated learningThe key insight: Good learners monitor their learning accurately.Poor learners overestimate their understanding (Dunning-Kruger adjacent).Score captures both learning quality AND monitoring accuracy of learning."""import kaggle_benchmarks as kbenchfrom dataclasses import dataclassimport numpy as npimport reimport json# Rule system generators defined above@dataclassclass LearnAndMonitor:    answer: str    learning_confidence: int  # 0-100: How well have you learned this system?    reasoning: strdef normalize_output(text: str) -> str:    text = text.strip().lower()    text = re.sub(r'\s+', ' ', text)    return textdef check_output(model_output: str, expected: str) -> bool:    m = normalize_output(model_output)    e = normalize_output(expected)    return e in m or m in edef goodman_kruskal_gamma(x: list, y: list) -> float:    n = len(x)    concordant = 0    discordant = 0    for i in range(n):        for j in range(i + 1, n):            product = (x[i] - x[j]) * (y[i] - y[j])            if product > 0:                concordant += 1            elif product < 0:                discordant += 1    denom = concordant + discordant    return (concordant - discordant) / denom if denom > 0 else 0.0# Generate systems for this benchmarkSYSTEMS = [    generate_symbol_system("crossdomain_sym_1", difficulty=2),    generate_number_system("crossdomain_num_1", difficulty=2),    generate_symbol_system("crossdomain_sym_2", difficulty=3),]@kbench.task(name="metacog_learning_monitoring")def metacog_learning_monitoring(llm) -> float:    """    Metacognitive Monitoring During Learning.    Tests whether models can accurately track their own learning progress.    Presents rules incrementally and measures both performance and    self-assessment at each stage.    Score = 0.30 * monitoring_gamma + 0.30 * learning_accuracy            + 0.20 * (1 - monitoring_bias) + 0.20 * learning_rate    Novel benchmark combining metacognition and learning tracks.    """    all_monitoring_confs = []    all_actual_accs = []    all_learning_accs = []    system_results = []    for system in SYSTEMS:        rules = system.rules        examples = system.examples        test_items = system.test_items        monitoring_confs = []  # Self-assessed learning quality        actual_accs = []       # Actual test accuracy at each stage        # Reveal rules incrementally: after 1, 2, ..., all rules        for stage in range(1, len(rules) + 1):            revealed_rules = rules[:stage]            # Also give some examples proportional to rules revealed            n_examples = min(stage * 2, len(examples))            revealed_examples = examples[:n_examples]            with kbench.chats.new(f"{system.name}_stage{stage}"):                # Build incremental learning prompt                prompt = f"You are learning the **{system.name}** rule system.\n\n"                prompt += f"**Rules learned so far ({stage}/{len(rules)}):**\n"                for r in revealed_rules:                    prompt += f"- {r}\n"                if revealed_examples:                    prompt += f"\n**Examples seen ({n_examples}):**\n"                    for ex in revealed_examples:                        prompt += f"  {ex['input']} → {ex['output']}\n"                # Test on first 3 test items                n_test = min(3, len(test_items))                stage_correct = 0                for ti in range(n_test):                    test_prompt = (                        prompt +                        f"\n**Test {ti+1}:** Apply what you've learned so far.\n"                        f"Input: {test_items[ti]['input']}\n\n"                        f"Also rate how confident you are (0-100) that you've "                        f"mastered the full rule system.\n\n"                        f"Respond with ONLY: {{\"answer\": \"<output>\", "                        f"\"learning_confidence\": <0-100>, "                        f"\"reasoning\": \"<brief>\"}}"                    )                    try:                        result = llm.prompt(test_prompt, schema=LearnAndMonitor)                        answer = result.answer                        conf = max(0, min(100, result.learning_confidence))                    except Exception:                        raw = llm.prompt(test_prompt)                        try:                            parsed = json.loads(re.search(r'\{.*\}', raw, re.DOTALL).group())                            answer = str(parsed.get("answer", raw))                            conf = max(0, min(100, int(parsed.get("learning_confidence", 50))))                        except Exception:                            answer = raw                            conf = 50                    if check_output(answer, test_items[ti]["output"]):                        stage_correct += 1                    monitoring_confs.append(conf)                stage_acc = stage_correct / n_test                # Replicate accuracy for each test item at this stage                for _ in range(n_test):                    actual_accs.append(stage_acc)        all_monitoring_confs.extend(monitoring_confs)        all_actual_accs.extend(actual_accs)        # Per-system learning accuracy (final stage)        if actual_accs:            all_learning_accs.append(actual_accs[-1])        system_results.append({            "name": system.name,            "difficulty": system.difficulty,            "monitoring_confs": monitoring_confs,            "actual_accs": actual_accs,            "n_stages": len(rules),        })    # ── Compute Metrics ──    # Monitoring accuracy: gamma between self-assessment and actual accuracy    gamma = goodman_kruskal_gamma(all_monitoring_confs,                                    [int(a * 100) for a in all_actual_accs])    # Monitoring bias: mean confidence - mean accuracy    mean_conf = np.mean(all_monitoring_confs) / 100    mean_acc = np.mean(all_actual_accs)    bias = abs(mean_conf - mean_acc)  # 0 = perfectly calibrated    # Learning accuracy (final stage, averaged across systems)    learning_acc = np.mean(all_learning_accs) if all_learning_accs else 0    # Learning rate: improvement from first to last stage    # Average across systems    learning_rates = []    for sr in system_results:        accs = sr["actual_accs"]        if len(accs) >= 2:            # Compare first stage mean to last stage mean            n_test = 3            first_acc = np.mean(accs[:n_test]) if len(accs) >= n_test else accs[0]            last_acc = np.mean(accs[-n_test:]) if len(accs) >= n_test else accs[-1]            learning_rates.append(max(0, last_acc - first_acc))    mean_lr = np.mean(learning_rates) if learning_rates else 0    gamma_norm = (gamma + 1) / 2    score = round(        0.30 * gamma_norm + 0.30 * float(learning_acc)        + 0.20 * (1 - min(1, bias)) + 0.20 * float(mean_lr),        4    )    # ── Logging ──    print(f"\n{'='*60}")    print(f"METACOGNITIVE MONITORING DURING LEARNING")    print(f"{'='*60}")    print(f"Systems tested: {len(SYSTEMS)}")    for sr in system_results:        print(f"\n--- {sr['name']} (difficulty={sr['difficulty']}) ---")        n_test = 3        for stage in range(sr["n_stages"]):            start = stage * n_test            end = start + n_test            stage_confs = sr["monitoring_confs"][start:end]            stage_acc = sr["actual_accs"][start] if start < len(sr["actual_accs"]) else 0            mean_c = np.mean(stage_confs) if stage_confs else 0            print(f"  Stage {stage+1}/{sr['n_stages']}: "                  f"acc={stage_acc:.2%}, self-assess={mean_c:.0f}%")    print(f"\n--- Aggregate Metrics ---")    print(f"Monitoring gamma:     {gamma:+.4f}")    print(f"Monitoring bias:      {bias:.3f} (|conf - acc|)")    print(f"Mean confidence:      {mean_conf:.2%}")    print(f"Mean accuracy:        {mean_acc:.2%}")    print(f"Learning accuracy:    {learning_acc:.2%}")    print(f"Mean learning rate:   {mean_lr:.3f}")    print(f"Composite score:      {score:.4f}")    return score# ─── Run ────────────────────────────────────────────────────────────metacog_learning_monitoring.run(llm=kbench.llm)